# News Topic Evolution and Trend Emergence Analytics Dashboard using Topic Modeling

#### Import Required Libraries and Dependencies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic
from sklearn.model_selection import train_test_split


C:\Users\harsh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Load Dataset and Perform Initial Data Preparation


In [2]:

df = pd.read_excel('Project_GT.xlsx')
df = df[['headline', 'short_description', 'category', 'date']]
df = df.dropna(subset=['headline'])
df['short_description'] = df['short_description'].fillna('')
df['text'] = df['headline'].astype(str) + ' ' + df['short_description'].astype(str)
df['date'] = pd.to_datetime(df['date'])



#### Clean and Preprocess News Articles for Topic Modeling

In [3]:

import re
import nltk
import spacy
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### Generate Topics Using BERTopic

In [ ]:
import pandas as pd
from bertopic import BERTopic
df = pd.read_pickle("cleaned_news_dataset.pkl")
df = df.dropna(subset=["clean_text"])
sample_df = df.sample(n=50000, random_state=42)
sample_df = sample_df.reset_index(drop=True)
docs = sample_df["clean_text"].tolist()
model = BERTopic(
    language="english",
    calculate_probabilities=False,
    verbose=True
)
topics, probabilities = model.fit_transform(docs)
sample_df["topic"] = topics
sample_df.to_pickle("news_with_topics_50k.pkl")
topic_info = model.get_topic_info()
print(topic_info.head())

2026-06-16 02:47:34,570 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 1563/1563 [02:07<00:00, 12.24it/s]
2026-06-16 02:49:47,271 - BERTopic - Embedding - Completed ✓
2026-06-16 02:49:47,271 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-16 02:49:53,941 - BERTopic - Dimensionality - Completed ✓
2026-06-16 02:49:53,943 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-16 02:49:55,451 - BERTopic - Cluster - Completed ✓
2026-06-16 02:49:55,459 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-16 02:49:56,204 - BERTopic - Representation - Completed ✓


   Topic  Count                             Name  \
0     -1  25455            -1_not_like_new_photo   
1      0   1056       0_gun_police_shoot_officer   
2      1    559      1_fashion_style_dress_photo   
3      2    531     2_gay_lgbt_transgender_pride   
4      3    485  3_wedding_bride_proposal_couple   

                                      Representation  \
0  [not, like, new, photo, one, make, get, trump,...   
1  [gun, police, shoot, officer, shooting, cop, k...   
2  [fashion, style, dress, photo, tumblr, week, w...   
3  [gay, lgbt, transgender, pride, queer, lgbtq, ...   
4  [wedding, bride, proposal, couple, groom, gues...   

                                 Representative_Docs  
0  [thing we ve learn sleep past year people reme...  
1  [texas police officer fatally shoot traffic st...  
2  [new york fashion week really like go runway s...  
3  [antigay bullying accept lgbt child enough par...  
4  [there s nothing wrong wear black wedding rule...  


#### Create Monthly Time Periods for Trend Analysis

In [19]:
sample_df["date"] = pd.to_datetime(sample_df["date"])
sample_df["month"] = sample_df["date"].dt.to_period("M")
print(sample_df[["date", "month"]].head())



         date    month
1  2017-09-08  2017-09
2  2016-12-12  2016-12
4  2016-02-10  2016-02
8  2015-06-02  2015-06
10 2015-01-23  2015-01


#### Analyze Topic Trends Over Time

In [20]:
topic_trends = (
    sample_df
    .groupby(["month", "topic"])
    .size()
    .reset_index(name="article_count")
)

#### Generate Topic Metadata and Export Final Files

In [22]:
sample_df = sample_df[sample_df["topic"] != -1]
topic_info = model.get_topic_info()
print(topic_info.head())
topic_info = topic_info[topic_info["Topic"] != -1]
topic_info.to_csv("topic_info.csv", index=False)



   Topic  Count                             Name  \
0     -1  25455            -1_not_like_new_photo   
1      0   1056       0_gun_police_shoot_officer   
2      1    559      1_fashion_style_dress_photo   
3      2    531     2_gay_lgbt_transgender_pride   
4      3    485  3_wedding_bride_proposal_couple   

                                      Representation  \
0  [not, like, new, photo, one, make, get, trump,...   
1  [gun, police, shoot, officer, shooting, cop, k...   
2  [fashion, style, dress, photo, tumblr, week, w...   
3  [gay, lgbt, transgender, pride, queer, lgbtq, ...   
4  [wedding, bride, proposal, couple, groom, gues...   

                                 Representative_Docs  
0  [thing we ve learn sleep past year people reme...  
1  [texas police officer fatally shoot traffic st...  
2  [new york fashion week really like go runway s...  
3  [antigay bullying accept lgbt child enough par...  
4  [there s nothing wrong wear black wedding rule...  
